# Module 02 — Supervised Learning
## 02-05: Support Vector Machines

**Objective:** Implement maximum-margin SVMs from scratch — primal subgradient (linear) and dual gradient ascent (kernel) — and understand why the kernel trick enables non-linear classification without explicit feature maps.

**Prerequisites:** 02-01 (Linear Regression), 02-02 (Logistic Regression & Binary Classification), 01-06 (Linear Algebra — dot products, projections), 01-09 (Calculus & Optimization — gradient descent).

**GPU Required:** No (classical ML)

## Part 0 — Setup & Prerequisites

Support Vector Machines (SVMs) find the decision boundary that maximises the geometric margin between classes. We implement two flavours: a **linear SVM** (primal subgradient descent on the hinge-loss objective) and a **kernel SVM** (projected gradient ascent on the dual Lagrangian). Both are tested on `make_moons` and `make_circles` — two non-linearly separable toy datasets that expose the need for the kernel trick.

In [ ]:
# ── Imports ──────────────────────────────────────────────────────────────────
import sys
import warnings
warnings.filterwarnings("ignore")

import random
import math
import time
from typing import Callable

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import torch

from sklearn.datasets import make_moons, make_circles
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score, f1_score, classification_report,
    confusion_matrix, ConfusionMatrixDisplay,
)
from sklearn.svm import SVC, LinearSVC
import sklearn

print(f"Python:       {sys.version.split()[0]}")
print(f"NumPy:        {np.__version__}")
print(f"Scikit-learn: {sklearn.__version__}")
print(f"PyTorch:      {torch.__version__}")
if torch.cuda.is_available():
    print(f"CUDA: {torch.version.cuda}")
    print(f"GPU:  {torch.cuda.get_device_name(0)}")

In [ ]:
# ── Reproducibility ───────────────────────────────────────────────────────────
SEED = 1103
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# Classical ML — no GPU needed
print(f"Random seed set: {SEED}")

In [ ]:
# ── Configuration ─────────────────────────────────────────────────────────────
# Dataset
N_SAMPLES:     int   = 400
NOISE_MOONS:   float = 0.15
NOISE_CIRCLES: float = 0.10
TEST_SIZE:     float = 0.20

# Linear SVM (Primal Subgradient)
C_LINEAR:      float = 1.0
LR_LINEAR:     float = 0.005
N_ITER_LINEAR: int   = 3000

# Kernel SVM (Dual Gradient Ascent)
C_KERNEL:    float = 1.0
GAMMA_RBF:   float = 1.0
DEGREE_POLY: int   = 3
COEF0_POLY:  float = 1.0
LR_DUAL:     float = 0.01
N_ITER_DUAL: int   = 800

### Data Loading & EDA

We use two synthetic 2-D datasets chosen for their non-linear structure:
- **make_moons** — two interleaved half-circles (mild non-linearity).
- **make_circles** — concentric rings (radial structure, not linearly separable).

Both datasets are small (400 samples) so our from-scratch solvers run in seconds.

In [ ]:
# ── Generate datasets ─────────────────────────────────────────────────────────
X_moons,   y_moons   = make_moons(
    n_samples=N_SAMPLES, noise=NOISE_MOONS,   random_state=SEED)
X_circles, y_circles = make_circles(
    n_samples=N_SAMPLES, noise=NOISE_CIRCLES, factor=0.5, random_state=SEED)

X_moon_train, X_moon_test, y_moon_train, y_moon_test = train_test_split(
    X_moons, y_moons, test_size=TEST_SIZE, random_state=SEED, stratify=y_moons)
X_circ_train, X_circ_test, y_circ_train, y_circ_test = train_test_split(
    X_circles, y_circles, test_size=TEST_SIZE, random_state=SEED, stratify=y_circles)

print("make_moons")
print(f"  Samples: {N_SAMPLES}  |  Train: {len(X_moon_train)}  |  Test: {len(X_moon_test)}")
print(f"  Features: {X_moons.shape[1]}")
print(f"  Class dist: {dict(zip(*np.unique(y_moons, return_counts=True)))}")
print()
print("make_circles")
print(f"  Samples: {N_SAMPLES}  |  Train: {len(X_circ_train)}  |  Test: {len(X_circ_test)}")
print(f"  Features: {X_circles.shape[1]}")
print(f"  Class dist: {dict(zip(*np.unique(y_circles, return_counts=True)))}")

# Majority-class baseline
baseline_moons   = max(np.mean(y_moon_test == 0), np.mean(y_moon_test == 1))
baseline_circles = max(np.mean(y_circ_test == 0), np.mean(y_circ_test == 1))
print(f"\nBaseline accuracy (majority class): moons={baseline_moons:.2%}, circles={baseline_circles:.2%}")

# EDA visualisation
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
colors = ["steelblue", "darkorange"]

for ax, X, y, title in [
    (axes[0], X_moons,   y_moons,   "make_moons"),
    (axes[1], X_circles, y_circles, "make_circles"),
]:
    for cls in [0, 1]:
        mask = y == cls
        ax.scatter(X[mask, 0], X[mask, 1], c=colors[cls], alpha=0.65,
                   s=18, label=f"Class {cls}")
    ax.set_xlabel("Feature 0", fontsize=10)
    ax.set_ylabel("Feature 1", fontsize=10)
    ax.set_title(title, fontsize=11)
    ax.legend(fontsize=9)

plt.suptitle("Classification Datasets", fontsize=12)
plt.tight_layout()
plt.show()

---
## Part 1 — Maximum-Margin Classifiers from Scratch

### The Maximum Margin Idea

Given labelled data $\{(\mathbf{x}_i, y_i)\}_{i=1}^n$ with $y_i \in \{-1, +1\}$,
a linear classifier predicts $\hat{y} = \text{sign}(\mathbf{w}^\top \mathbf{x} + b)$.
The **geometric margin** of the $i$-th point is:

$$\gamma_i = \frac{y_i(\mathbf{w}^\top \mathbf{x}_i + b)}{\|\mathbf{w}\|}$$

The SVM finds the $\mathbf{w}$ and $b$ that **maximise the minimum margin**
$\min_i \gamma_i$. After normalising so that the margin equals $1/\|\mathbf{w}\|$,
this is equivalent to:

$$\min_{\mathbf{w},b} \tfrac{1}{2}\|\mathbf{w}\|^2 \quad \text{s.t.} \quad y_i(\mathbf{w}^\top \mathbf{x}_i + b) \ge 1 \; \forall i$$

For **soft-margin** (non-separable data) we introduce slack variables $\xi_i \ge 0$:

$$\min_{\mathbf{w},b,\boldsymbol{\xi}} \tfrac{1}{2}\|\mathbf{w}\|^2 + C \sum_i \xi_i \quad \text{s.t.} \quad y_i(\mathbf{w}^\top \mathbf{x}_i + b) \ge 1 - \xi_i, \; \xi_i \ge 0$$

Substituting the optimal slack $\xi_i = \max(0, 1 - y_i f(\mathbf{x}_i))$ yields
the unconstrained **hinge-loss + L2 primal**:

$$\min_{\mathbf{w},b} \underbrace{\tfrac{1}{2}\|\mathbf{w}\|^2}_{\text{margin regulariser}} + C \underbrace{\sum_i \max(0, 1 - y_i(\mathbf{w}^\top \mathbf{x}_i + b))}_{\text{hinge loss}}$$

We optimise this primal with **subgradient descent**.

In [ ]:
# ── Margin geometry: hard-margin SVM illustrated on toy separable data ─────────
rng = np.random.default_rng(SEED)

n_toy = 30
X_toy_pos = rng.multivariate_normal([ 2.0,  1.0], [[0.3, 0], [0, 0.3]], n_toy)
X_toy_neg = rng.multivariate_normal([-1.0, -1.0], [[0.3, 0], [0, 0.3]], n_toy)
X_toy = np.vstack([X_toy_pos, X_toy_neg])
y_toy_signed = np.array([1] * n_toy + [-1] * n_toy)
y_toy_01 = (y_toy_signed + 1) // 2  # {-1,+1} -> {0,1}

# Use sklearn SVC (large C -> hard-margin solution) just to get exact hyperplane
from sklearn.svm import SVC as _SVC
_clf_toy = _SVC(kernel="linear", C=1e6)
_clf_toy.fit(X_toy, y_toy_signed)
w_toy = _clf_toy.coef_[0]
b_toy = float(_clf_toy.intercept_[0])


def hyperplane_y(
    x: np.ndarray,
    w: np.ndarray,
    b: float,
    offset: float = 0.0,
) -> np.ndarray:
    """Compute y-coordinates on the hyperplane w[0]*x + w[1]*y + b = offset.

    Args:
        x:      x-coordinates, shape (n,).
        w:      Weight vector [w0, w1].
        b:      Bias scalar.
        offset: Right-hand-side offset (0 = decision, +/-1 = margin planes).

    Returns:
        y-coordinates, shape (n,).
    """
    return -(w[0] * x + b - offset) / (w[1] + 1e-12)


x_min_toy = X_toy[:, 0].min() - 0.5
x_max_toy = X_toy[:, 0].max() + 0.5
x_plot = np.linspace(x_min_toy, x_max_toy, 200)

y_decision   = hyperplane_y(x_plot, w_toy, b_toy,  0.0)
y_margin_pos = hyperplane_y(x_plot, w_toy, b_toy,  1.0)
y_margin_neg = hyperplane_y(x_plot, w_toy, b_toy, -1.0)

margin_width = 2.0 / np.linalg.norm(w_toy)

fig, ax = plt.subplots(figsize=(8, 5))
ax.scatter(X_toy_pos[:, 0], X_toy_pos[:, 1],
           c="darkorange", s=45, label="Class +1", zorder=3)
ax.scatter(X_toy_neg[:, 0], X_toy_neg[:, 1],
           c="steelblue",  s=45, label="Class -1", zorder=3)

sv_idx = _clf_toy.support_
ax.scatter(X_toy[sv_idx, 0], X_toy[sv_idx, 1],
           s=180, facecolors="none", edgecolors="red", linewidths=2,
           label=f"Support vectors ({len(sv_idx)})", zorder=4)

ax.plot(x_plot, y_decision,   "k-",  linewidth=2.5, label="Decision boundary")
ax.plot(x_plot, y_margin_pos, "r--", linewidth=1.5, label="Margin hyperplane +1")
ax.plot(x_plot, y_margin_neg, "b--", linewidth=1.5, label="Margin hyperplane -1")
ax.fill_between(x_plot, y_margin_neg, y_margin_pos,
                alpha=0.12, color="gray", label="Margin band")

ax.set_title(
    f"Hard-Margin SVM — Geometric Margin = {margin_width:.3f}  |  "
    f"Support Vectors = {len(sv_idx)}", fontsize=11)
ax.set_xlabel("Feature 0")
ax.set_ylabel("Feature 1")
ax.legend(fontsize=9, loc="upper left")
ax.set_xlim(x_min_toy, x_max_toy)
plt.tight_layout()
plt.show()

print(f"Decision boundary: {w_toy[0]:.3f}*x0 + {w_toy[1]:.3f}*x1 + {b_toy:.3f} = 0")
print(f"||w|| = {np.linalg.norm(w_toy):.4f}   margin = 2/||w|| = {margin_width:.4f}")
print("Only circled points (support vectors) determine the decision boundary.")

### 1.1 Hinge Loss and Subgradient

The **hinge loss** for sample $(\mathbf{x}_i, y_i)$ is:

$$\ell_i = \max(0,\; 1 - y_i(\mathbf{w}^\top \mathbf{x}_i + b))$$

Zero when correctly classified with margin $\ge 1$; grows linearly otherwise.
The **subgradient** (hinge is non-smooth at margin $= 1$) is:

$$\frac{\partial \ell_i}{\partial \mathbf{w}} = \begin{cases}
-y_i \mathbf{x}_i & \text{if } y_i f_i < 1 \quad \text{(violator)} \\
\mathbf{0} & \text{otherwise}
\end{cases}$$

Full primal subgradient:

$$\nabla_{\mathbf{w}} \mathcal{L} = \mathbf{w} - \frac{C}{n} \sum_{i \in \text{violators}} y_i \mathbf{x}_i$$

In [ ]:
# ── Hinge Loss and Subgradient ────────────────────────────────────────────────

def compute_hinge_loss(
    w: np.ndarray,
    b: float,
    X: np.ndarray,
    y_signed: np.ndarray,
    C: float,
) -> float:
    """Compute the soft-margin SVM primal objective.

    Primal: (1/2)||w||^2 + C * mean(max(0, 1 - y_i*(w^T x_i + b)))

    Args:
        w:        Weight vector, shape (n_features,).
        b:        Bias scalar.
        X:        Feature matrix, shape (n_samples, n_features).
        y_signed: Labels in {-1, +1}, shape (n_samples,).
        C:        Regularisation constant (larger => penalise violations more).

    Returns:
        Scalar primal objective value.
    """
    margins = y_signed * (X @ w + b)
    hinge   = np.maximum(0.0, 1.0 - margins)
    return float(0.5 * np.dot(w, w) + C * hinge.mean())


def compute_linear_svm_gradient(
    w: np.ndarray,
    b: float,
    X: np.ndarray,
    y_signed: np.ndarray,
    C: float,
) -> tuple[np.ndarray, float]:
    """Compute the subgradient of the soft-margin SVM primal objective.

    For samples with margin < 1 (violators), the hinge subgradient is -y_i x_i.
    Samples with margin >= 1 contribute zero to the hinge subgradient.

    Args:
        w:        Weight vector, shape (n_features,).
        b:        Bias scalar.
        X:        Feature matrix, shape (n_samples, n_features).
        y_signed: Labels in {-1, +1}, shape (n_samples,).
        C:        Regularisation constant.

    Returns:
        Tuple of (grad_w, grad_b) — subgradients w.r.t. w and b.
    """
    n = len(y_signed)
    margins  = y_signed * (X @ w + b)
    violated = margins < 1.0

    if violated.any():
        grad_w = w - (C / n) * np.sum(
            y_signed[violated, np.newaxis] * X[violated], axis=0)
        grad_b = -(C / n) * np.sum(y_signed[violated])
    else:
        grad_w = w.copy()
        grad_b = 0.0

    return grad_w, float(grad_b)


# Hinge loss sanity check
w_test    = np.array([1.0, 0.0])
b_test    = 0.0
X_check   = np.array([[1.5, 0.0], [-1.5, 0.0], [0.2, 0.0]])
y_check   = np.array([1.0, -1.0, 1.0])
margins_c = y_check * (X_check @ w_test + b_test)
hinge_c   = np.maximum(0.0, 1.0 - margins_c)
print("Margins:", np.round(margins_c, 2), " (>= 1 = correct)")
print("Hinge:  ", np.round(hinge_c,   2), " (0 = correct, > 0 = violated)")
print(f"Primal loss (C=1): {compute_hinge_loss(w_test, b_test, X_check, y_check, 1.0):.4f}")

### 1.2 The Kernel Trick

A linear SVM fails when classes are not linearly separable. One solution maps
$\mathbf{x}$ to a high-dimensional feature space $\phi(\mathbf{x})$ where they
become separable. The **kernel trick** replaces all inner products
$\phi(\mathbf{x}_i)^\top \phi(\mathbf{x}_j)$ with a kernel function
$k(\mathbf{x}_i, \mathbf{x}_j)$ evaluated in the original space:

| Kernel | Formula | Intuition |
|--------|---------|-----------|
| Linear | $k(\mathbf{x}, \mathbf{x}') = \mathbf{x}^\top \mathbf{x}'$ | No transformation |
| RBF (Gaussian) | $k(\mathbf{x}, \mathbf{x}') = \exp(-\gamma \|\mathbf{x} - \mathbf{x}'\|^2)$ | Infinite-dimensional Gaussian |
| Polynomial | $k(\mathbf{x}, \mathbf{x}') = (\mathbf{x}^\top \mathbf{x}' + c)^d$ | Degree-$d$ polynomial features |

The **dual Lagrangian** (from KKT conditions on the soft-margin primal) only
involves kernel evaluations:

$$\max_{\boldsymbol{\alpha}} \sum_i \alpha_i - \frac{1}{2} \sum_{i,j} \alpha_i \alpha_j y_i y_j k(\mathbf{x}_i, \mathbf{x}_j)$$
$$\text{s.t.} \quad 0 \le \alpha_i \le C, \quad \sum_i \alpha_i y_i = 0$$

Points with $\alpha_i > 0$ are **support vectors** — only they influence predictions,
giving SVMs their sparsity property.

In [ ]:
# ── Kernel Functions ──────────────────────────────────────────────────────────

def linear_kernel(X1: np.ndarray, X2: np.ndarray) -> np.ndarray:
    """Compute the linear kernel matrix: K_ij = x_i^T x_j.

    Args:
        X1: Query matrix, shape (m, d).
        X2: Reference matrix, shape (n, d).

    Returns:
        Kernel matrix, shape (m, n).
    """
    return X1 @ X2.T


def rbf_kernel(
    X1: np.ndarray,
    X2: np.ndarray,
    gamma: float = 1.0,
) -> np.ndarray:
    """Compute the RBF (Gaussian) kernel matrix.

    K_ij = exp(-gamma * ||x_i - x_j||^2)

    Uses identity ||x_i - x_j||^2 = ||x_i||^2 + ||x_j||^2 - 2 x_i^T x_j
    to avoid materialising the full pairwise difference tensor.

    Args:
        X1:    Query matrix, shape (m, d).
        X2:    Reference matrix, shape (n, d).
        gamma: Bandwidth parameter. Larger => narrower Gaussians.

    Returns:
        Kernel matrix, shape (m, n).
    """
    sq_norms_1 = np.sum(X1 ** 2, axis=1, keepdims=True)
    sq_norms_2 = np.sum(X2 ** 2, axis=1, keepdims=True)
    sq_dists   = sq_norms_1 + sq_norms_2.T - 2.0 * (X1 @ X2.T)
    return np.exp(-gamma * np.maximum(sq_dists, 0.0))


def poly_kernel(
    X1: np.ndarray,
    X2: np.ndarray,
    degree: int = 3,
    coef0: float = 1.0,
) -> np.ndarray:
    """Compute the polynomial kernel matrix.

    K_ij = (x_i^T x_j + coef0)^degree

    Args:
        X1:     Query matrix, shape (m, d).
        X2:     Reference matrix, shape (n, d).
        degree: Polynomial degree.
        coef0:  Additive constant (controls low-degree term influence).

    Returns:
        Kernel matrix, shape (m, n).
    """
    return (X1 @ X2.T + coef0) ** degree


def compute_kernel_matrix(
    X1: np.ndarray,
    X2: np.ndarray,
    kernel: str = "rbf",
    gamma: float = 1.0,
    degree: int = 3,
    coef0: float = 1.0,
) -> np.ndarray:
    """Dispatch to the appropriate kernel function by name.

    Args:
        X1:     Query matrix, shape (m, d).
        X2:     Reference matrix, shape (n, d).
        kernel: One of 'linear', 'rbf', 'poly'.
        gamma:  RBF bandwidth (used when kernel='rbf').
        degree: Polynomial degree (used when kernel='poly').
        coef0:  Polynomial constant (used when kernel='poly').

    Returns:
        Kernel matrix, shape (m, n).

    Raises:
        ValueError: If kernel name is not recognised.
    """
    if kernel == "linear":
        return linear_kernel(X1, X2)
    elif kernel == "rbf":
        return rbf_kernel(X1, X2, gamma=gamma)
    elif kernel == "poly":
        return poly_kernel(X1, X2, degree=degree, coef0=coef0)
    else:
        raise ValueError(
            f"Unknown kernel '{kernel}'. Choose 'linear', 'rbf', or 'poly'.")


# Verify RBF kernel: symmetric, PSD, unit diagonal
X_verify    = np.random.randn(10, 2)
K_verify    = rbf_kernel(X_verify, X_verify, gamma=1.0)
eigenvalues = np.linalg.eigvalsh(K_verify)
print(f"RBF kernel matrix shape:    {K_verify.shape}")
print(f"Symmetric:                  {np.allclose(K_verify, K_verify.T)}")
print(f"PSD (all eigenvalues >= 0): {np.all(eigenvalues >= -1e-10)}")
print(f"Unit diagonal (k(x,x)=1):   {np.allclose(np.diag(K_verify), 1.0)}")

In [ ]:
# ── Kernel Gram Matrix Structure ──────────────────────────────────────────────
# Show that the RBF kernel makes make_circles separable in kernel space
# (block-diagonal structure when rows/cols are sorted by class label)

fig, axes = plt.subplots(1, 3, figsize=(14, 4))

K_linear_circ = linear_kernel(X_circ_train, X_circ_train)
K_rbf_circ    = rbf_kernel(X_circ_train, X_circ_train, gamma=GAMMA_RBF)
K_rbf_moon    = rbf_kernel(X_moon_train, X_moon_train, gamma=GAMMA_RBF)

sort_circ = np.argsort(y_circ_train)
sort_moon = np.argsort(y_moon_train)

for ax, K, sort_idx, title in [
    (axes[0], K_linear_circ, sort_circ,
     "Linear kernel — circles\n(no block structure -> not separable)"),
    (axes[1], K_rbf_circ,    sort_circ,
     f"RBF kernel — circles\ngamma={GAMMA_RBF}"),
    (axes[2], K_rbf_moon,    sort_moon,
     f"RBF kernel — moons\ngamma={GAMMA_RBF}"),
]:
    K_sorted = K[sort_idx][:, sort_idx]
    im = ax.imshow(K_sorted, cmap="viridis", aspect="auto")
    plt.colorbar(im, ax=ax, shrink=0.8)
    ax.set_title(title, fontsize=10)
    ax.set_xlabel("Sample (sorted by class)")
    ax.set_ylabel("Sample (sorted by class)")

plt.suptitle("Kernel Gram Matrix — Block Structure Reveals Separability", fontsize=12)
plt.tight_layout()
plt.show()

print("Interpretation:")
print("  Linear kernel on circles: no block structure -> classes not linearly separable.")
print("  RBF kernel on circles: 2×2 block structure -> separable in RKHS.")
print("  RBF kernel on moons: partial block structure (the two moons interleave).")

---
## Part 2 — Putting It All Together

### 2.1 Linear SVM (Primal Subgradient)

We combine `compute_hinge_loss` and `compute_linear_svm_gradient` into a full
`LinearSVMScratch` class. The `fit` method runs subgradient descent for
`n_iterations` steps, tracking the primal objective to verify convergence.

In [ ]:
class LinearSVMScratch:
    """Soft-margin linear SVM trained via primal subgradient descent.

    Optimises the hinge-loss primal:
        min_{w,b} (1/2)||w||^2 + C * mean(max(0, 1 - y_i*(w^T x_i + b)))

    Attributes:
        C:             Soft-margin regularisation strength.
        learning_rate: Initial step size for subgradient descent.
        n_iterations:  Number of gradient steps.
        weights_:      Fitted weight vector, shape (n_features,).
        bias_:         Fitted bias scalar.
        losses_:       Primal objective sampled every 50 steps.
    """

    def __init__(
        self,
        C: float = 1.0,
        learning_rate: float = 0.005,
        n_iterations: int = 3000,
    ) -> None:
        """Initialise LinearSVMScratch with soft-margin hyperparameters.

        Args:
            C:             Trade-off between margin width and slack violations.
                           Larger C => smaller margin, fewer violations allowed.
            learning_rate: Initial step size; decays by 0.9 every 500 steps.
            n_iterations:  Total number of subgradient steps.
        """
        self.C             = C
        self.learning_rate = learning_rate
        self.n_iterations  = n_iterations
        self.weights_: np.ndarray = np.array([])
        self.bias_: float         = 0.0
        self.losses_: list[float] = []

    def fit(self, X: np.ndarray, y: np.ndarray) -> "LinearSVMScratch":
        """Fit the linear SVM via subgradient descent on the primal objective.

        Args:
            X: Feature matrix, shape (n_samples, n_features).
            y: Binary labels in {0, 1}, shape (n_samples,).

        Returns:
            self — the fitted classifier.
        """
        n_samples, n_features = X.shape
        y_signed = 2 * y - 1  # {0, 1} -> {-1, +1}

        self.weights_ = np.zeros(n_features)
        self.bias_    = 0.0
        self.losses_  = []
        lr = self.learning_rate

        for step in range(self.n_iterations):
            if step % 50 == 0:
                self.losses_.append(
                    compute_hinge_loss(self.weights_, self.bias_, X, y_signed, self.C))

            grad_w, grad_b = compute_linear_svm_gradient(
                self.weights_, self.bias_, X, y_signed, self.C)

            self.weights_ -= lr * grad_w
            self.bias_    -= lr * grad_b

            if (step + 1) % 500 == 0:
                lr *= 0.9

        return self

    def decision_function(self, X: np.ndarray) -> np.ndarray:
        """Compute raw SVM scores (signed distance to decision boundary).

        Args:
            X: Feature matrix, shape (n_samples, n_features).

        Returns:
            Score array, shape (n_samples,). Positive => predict class 1.
        """
        return X @ self.weights_ + self.bias_

    def predict(self, X: np.ndarray) -> np.ndarray:
        """Predict binary class labels.

        Args:
            X: Feature matrix, shape (n_samples, n_features).

        Returns:
            Predicted labels in {0, 1}, shape (n_samples,).
        """
        return (self.decision_function(X) >= 0).astype(int)

    def score(self, X: np.ndarray, y: np.ndarray) -> float:
        """Compute accuracy on the given data.

        Args:
            X: Feature matrix, shape (n_samples, n_features).
            y: True labels in {0, 1}, shape (n_samples,).

        Returns:
            Accuracy in [0, 1].
        """
        return float(np.mean(self.predict(X) == y))

    def margin_width(self) -> float:
        """Return the geometric margin width 2 / ||w||.

        Returns:
            Geometric margin width. Larger is better (hard-margin case).
        """
        norm_w = np.linalg.norm(self.weights_)
        return float(2.0 / norm_w) if norm_w > 1e-12 else float("inf")

    def __repr__(self) -> str:
        return (f"LinearSVMScratch(C={self.C}, "
                f"learning_rate={self.learning_rate}, "
                f"n_iterations={self.n_iterations})")

### 2.2 Kernel SVM (Dual Projected Gradient Ascent)

For non-linear kernels we optimise the **dual Lagrangian**. The gradient of the
dual with respect to $\alpha_i$ is:

$$\frac{\partial W}{\partial \alpha_i} = 1 - \sum_j \alpha_j y_j k(\mathbf{x}_i, \mathbf{x}_j) y_i = 1 - (\mathbf{Q}\boldsymbol{\alpha})_i$$

where $Q_{ij} = y_i y_j k(\mathbf{x}_i, \mathbf{x}_j)$.

**Algorithm — Projected Gradient Ascent:**
1. Gradient step: $\boldsymbol{\alpha} \leftarrow \boldsymbol{\alpha} + \eta (1 - \mathbf{Q}\boldsymbol{\alpha})$
2. Clip to box: $\alpha_i \leftarrow \text{clip}(\alpha_i, 0, C)$
3. Equality projection: $\boldsymbol{\alpha} \leftarrow \boldsymbol{\alpha} - \frac{\boldsymbol{\alpha}^\top \mathbf{y}}{n} \mathbf{y}$ (since $\|\mathbf{y}\|^2 = n$)
4. Re-clip to box (step 3 may push $\alpha_i$ slightly outside $[0, C]$)

**Prediction** uses only support vectors ($\alpha_i > 0$):
$$f(\mathbf{x}) = \sum_i \alpha_i y_i k(\mathbf{x}_i, \mathbf{x}) + b$$

In [ ]:
class KernelSVMScratch:
    """Kernel SVM trained via projected gradient ascent on the dual Lagrangian.

    Supports linear, RBF, and polynomial kernels. Support vectors are the
    samples with dual variable alpha_i > threshold after convergence.

    Attributes:
        C:               Soft-margin regularisation constant.
        kernel:          Kernel type: 'linear', 'rbf', or 'poly'.
        gamma:           RBF bandwidth (used when kernel='rbf').
        degree:          Polynomial degree (used when kernel='poly').
        coef0:           Polynomial additive constant.
        learning_rate:   Initial step size for gradient ascent.
        n_iterations:    Number of optimisation steps.
        alpha_:          Fitted dual variables, shape (n_train,).
        bias_:           Fitted bias scalar.
        X_train_:        Training features stored for prediction.
        y_signed_:       Training labels in {-1, +1}.
        n_support_:      Number of support vectors.
        sv_mask_:        Boolean mask identifying support vectors.
        dual_objectives_: Recorded dual objective values.
    """

    def __init__(
        self,
        C: float = 1.0,
        kernel: str = "rbf",
        gamma: float = 1.0,
        degree: int = 3,
        coef0: float = 1.0,
        learning_rate: float = 0.01,
        n_iterations: int = 800,
    ) -> None:
        """Initialise KernelSVMScratch with kernel and optimisation hyperparameters.

        Args:
            C:            Trade-off between margin and violations. Larger => less regularisation.
            kernel:       Kernel function: 'linear', 'rbf', or 'poly'.
            gamma:        RBF bandwidth. Larger gamma => narrower Gaussians => more complex boundary.
            degree:       Polynomial kernel degree.
            coef0:        Polynomial kernel additive constant.
            learning_rate: Dual gradient ascent step size; decays every 200 steps.
            n_iterations: Total optimisation steps.
        """
        self.C             = C
        self.kernel        = kernel
        self.gamma         = gamma
        self.degree        = degree
        self.coef0         = coef0
        self.learning_rate = learning_rate
        self.n_iterations  = n_iterations
        self.alpha_: np.ndarray            = np.array([])
        self.bias_: float                  = 0.0
        self.X_train_: np.ndarray          = np.array([[]])
        self.y_signed_: np.ndarray         = np.array([])
        self.n_support_: int               = 0
        self.sv_mask_: np.ndarray          = np.array([], dtype=bool)
        self.dual_objectives_: list[float] = []

    def _compute_kernel_matrix(
        self,
        X1: np.ndarray,
        X2: np.ndarray,
    ) -> np.ndarray:
        """Compute the kernel matrix between two sample sets.

        Args:
            X1: Query matrix, shape (m, d).
            X2: Reference matrix, shape (n, d).

        Returns:
            Kernel matrix, shape (m, n).
        """
        return compute_kernel_matrix(
            X1, X2,
            kernel=self.kernel,
            gamma=self.gamma,
            degree=self.degree,
            coef0=self.coef0,
        )

    def fit(self, X: np.ndarray, y: np.ndarray) -> "KernelSVMScratch":
        """Fit the kernel SVM via projected gradient ascent on the dual.

        Dual problem: max_alpha sum(alpha) - 0.5 * alpha^T Q alpha
        Subject to:  0 <= alpha_i <= C, sum(alpha_i * y_i) = 0

        Args:
            X: Feature matrix, shape (n_samples, n_features).
            y: Binary labels in {0, 1}, shape (n_samples,).

        Returns:
            self — the fitted classifier.
        """
        n = len(y)
        y_signed = 2 * y - 1  # {0, 1} -> {-1, +1}

        # Precompute n×n kernel Gram matrix and Q_ij = y_i * y_j * K_ij
        K = self._compute_kernel_matrix(X, X)
        Q = (y_signed[:, np.newaxis] * y_signed[np.newaxis, :]) * K

        alpha = np.zeros(n)
        self.dual_objectives_ = []
        lr = self.learning_rate

        for step in range(self.n_iterations):
            # ── Gradient of dual objective: grad_i = 1 - (Q alpha)_i ─────────
            gradient = np.ones(n) - Q @ alpha

            # ── Gradient ascent step ──────────────────────────────────────────
            alpha = alpha + lr * gradient

            # ── Project onto box constraint: 0 <= alpha_i <= C ───────────────
            alpha = np.clip(alpha, 0.0, self.C)

            # ── Project onto equality constraint: sum(alpha * y) = 0 ─────────
            # Exact projection onto hyperplane {alpha: alpha^T y = 0}:
            # alpha <- alpha - (alpha^T y / ||y||^2) * y
            # Since y_i in {-1,+1} => ||y||^2 = n
            violation = float(np.dot(alpha, y_signed))
            alpha = alpha - (violation / n) * y_signed
            alpha = np.clip(alpha, 0.0, self.C)  # re-clip after projection

            # ── Learning rate decay every 200 steps ───────────────────────────
            if (step + 1) % 200 == 0:
                lr *= 0.9

            if step % 50 == 0:
                dual_val = float(np.sum(alpha) - 0.5 * float(alpha @ Q @ alpha))
                self.dual_objectives_.append(dual_val)

        # ── Identify support vectors ──────────────────────────────────────────
        sv_threshold = 1e-4 * self.C
        sv_mask = alpha > sv_threshold

        self.alpha_    = alpha
        self.sv_mask_  = sv_mask
        self.X_train_  = X
        self.y_signed_ = y_signed
        self.n_support_ = int(sv_mask.sum())

        # ── Estimate bias b ───────────────────────────────────────────────────
        # Margin SVs (0 < alpha < C) sit on the margin hyperplane:
        # y_i * f(x_i) = 1  =>  b = y_i - sum_j alpha_j y_j K(x_j, x_i)
        margin_sv = sv_mask & (alpha < self.C * (1.0 - 1e-4))
        ref_sv = margin_sv if margin_sv.any() else sv_mask

        if ref_sv.any():
            K_ref     = self._compute_kernel_matrix(X[ref_sv], X)
            decisions = K_ref @ (alpha * y_signed)
            self.bias_ = float(np.mean(y_signed[ref_sv] - decisions))
        else:
            self.bias_ = 0.0

        return self

    def decision_function(self, X: np.ndarray) -> np.ndarray:
        """Compute raw SVM scores for test samples.

        f(x) = sum_i alpha_i y_i K(x_i, x) + b

        Args:
            X: Feature matrix, shape (n_samples, n_features).

        Returns:
            Score array, shape (n_samples,). Positive => class 1.
        """
        K_test = self._compute_kernel_matrix(X, self.X_train_)
        return K_test @ (self.alpha_ * self.y_signed_) + self.bias_

    def predict(self, X: np.ndarray) -> np.ndarray:
        """Predict binary class labels.

        Args:
            X: Feature matrix, shape (n_samples, n_features).

        Returns:
            Predicted labels in {0, 1}, shape (n_samples,).
        """
        return (self.decision_function(X) >= 0).astype(int)

    def score(self, X: np.ndarray, y: np.ndarray) -> float:
        """Compute accuracy on the given data.

        Args:
            X: Feature matrix, shape (n_samples, n_features).
            y: True labels in {0, 1}, shape (n_samples,).

        Returns:
            Accuracy in [0, 1].
        """
        return float(np.mean(self.predict(X) == y))

    def __repr__(self) -> str:
        return (f"KernelSVMScratch(C={self.C}, kernel=\'{self.kernel}\', "
                f"gamma={self.gamma}, n_iter={self.n_iterations})")

In [ ]:
# ── Decision Boundary Visualisation Helper ────────────────────────────────────

def plot_decision_boundary(
    model: object,
    X: np.ndarray,
    y: np.ndarray,
    ax: plt.Axes,
    title: str = "",
    h: float = 0.04,
    show_sv: bool = False,
) -> None:
    """Plot a 2-D binary decision boundary for any classifier with predict().

    Rasterises the classifier over a meshgrid, fills background colours,
    draws the decision boundary contour, and overlays the training data.

    Args:
        model:   Fitted classifier with a predict(X) -> np.ndarray method.
        X:       Training features, shape (n_samples, 2).
        y:       Training labels in {0, 1}, shape (n_samples,).
        ax:      Matplotlib Axes object to draw on.
        title:   Title string for the axes.
        h:       Meshgrid step size (smaller => finer, slower).
        show_sv: If True and model has sv_mask_, highlight support vectors.
    """
    x_min, x_max = X[:, 0].min() - 0.3, X[:, 0].max() + 0.3
    y_min, y_max = X[:, 1].min() - 0.3, X[:, 1].max() + 0.3
    xx, yy = np.meshgrid(
        np.arange(x_min, x_max, h),
        np.arange(y_min, y_max, h),
    )
    grid = np.column_stack([xx.ravel(), yy.ravel()])
    Z = model.predict(grid).reshape(xx.shape)

    ax.contourf(xx, yy, Z, alpha=0.25, cmap="coolwarm", levels=[-0.5, 0.5, 1.5])
    ax.contour(xx, yy, Z, colors="k", linewidths=1.2, levels=[0.5])

    palette = ["steelblue", "darkorange"]
    for cls in [0, 1]:
        mask = y == cls
        ax.scatter(X[mask, 0], X[mask, 1], c=palette[cls],
                   s=18, alpha=0.75, zorder=2)

    if show_sv and hasattr(model, "sv_mask_") and model.sv_mask_.any():
        ax.scatter(
            X[model.sv_mask_, 0], X[model.sv_mask_, 1],
            s=100, facecolors="none", edgecolors="red",
            linewidths=1.5, zorder=3, label="Support vectors",
        )
        ax.legend(fontsize=8)

    ax.set_title(title, fontsize=10)
    ax.set_xlabel("Feature 0", fontsize=9)
    ax.set_ylabel("Feature 1", fontsize=9)

### Part 2 — Sanity Checks

Before training on real data, we verify correctness on a perfectly linearly
separable toy problem. Both classifiers should achieve 100% training accuracy.

In [ ]:
# ── Sanity Check 1: LinearSVMScratch on separable toy data ───────────────────
toy_linear = LinearSVMScratch(C=1.0, learning_rate=0.01, n_iterations=2000)
toy_linear.fit(X_toy, y_toy_01)

print(f"LinearSVMScratch on separable toy: train acc = {toy_linear.score(X_toy, y_toy_01):.2%}")
print(f"  Margin width = {toy_linear.margin_width():.4f}")

# ── Sanity Check 2: KernelSVMScratch (RBF) on separable toy data ──────────────
toy_kernel = KernelSVMScratch(C=1.0, kernel="rbf", gamma=1.0,
                               learning_rate=0.01, n_iterations=800)
toy_kernel.fit(X_toy, y_toy_01)

print(f"KernelSVMScratch (RBF) on separable toy: train acc = {toy_kernel.score(X_toy, y_toy_01):.2%}")
print(f"  Support vectors: {toy_kernel.n_support_} / {len(y_toy_01)}")

# ── Sanity Check 3: Check prediction agreement ────────────────────────────────
agree = float(np.mean(toy_linear.predict(X_toy) == toy_kernel.predict(X_toy)))
print(f"  Prediction agreement (Linear vs Kernel): {agree:.2%}")

# Plot dual objective convergence on toy data
fig, axes = plt.subplots(1, 2, figsize=(12, 3))

axes[0].plot(toy_linear.losses_, color="steelblue", linewidth=2)
axes[0].set_xlabel("Recorded step")
axes[0].set_ylabel("Primal objective")
axes[0].set_title("LinearSVM — Primal Convergence (toy data)")

axes[1].plot(toy_kernel.dual_objectives_, color="darkorange", linewidth=2)
axes[1].set_xlabel("Recorded step")
axes[1].set_ylabel("Dual objective")
axes[1].set_title("KernelSVM — Dual Convergence (toy data)")

plt.tight_layout()
plt.show()

---
## Part 3 — Training on make_moons & make_circles

The linear SVM serves as a baseline (it will struggle on the non-linear
boundaries), while the kernel SVM with RBF should handle both datasets well.

In [ ]:
# ── Train Linear SVM on both datasets ────────────────────────────────────────
t0 = time.time()
linear_svm_moons = LinearSVMScratch(
    C=C_LINEAR, learning_rate=LR_LINEAR, n_iterations=N_ITER_LINEAR)
linear_svm_moons.fit(X_moon_train, y_moon_train)
t_linear_moons = time.time() - t0

acc_lin_moons_tr = linear_svm_moons.score(X_moon_train, y_moon_train)
acc_lin_moons_te = linear_svm_moons.score(X_moon_test,  y_moon_test)

t0 = time.time()
linear_svm_circles = LinearSVMScratch(
    C=C_LINEAR, learning_rate=LR_LINEAR, n_iterations=N_ITER_LINEAR)
linear_svm_circles.fit(X_circ_train, y_circ_train)
t_linear_circles = time.time() - t0

acc_lin_circ_tr = linear_svm_circles.score(X_circ_train, y_circ_train)
acc_lin_circ_te = linear_svm_circles.score(X_circ_test,  y_circ_test)

print("LinearSVMScratch — make_moons")
print(f"  Train: {acc_lin_moons_tr:.2%} | Test: {acc_lin_moons_te:.2%} | "
      f"Time: {t_linear_moons:.2f}s | Margin: {linear_svm_moons.margin_width():.4f}")
print()
print("LinearSVMScratch — make_circles")
print(f"  Train: {acc_lin_circ_tr:.2%} | Test: {acc_lin_circ_te:.2%} | "
      f"Time: {t_linear_circles:.2f}s | Margin: {linear_svm_circles.margin_width():.4f}")
print("  (Poor circles accuracy expected: the boundary is circular, not linear.)")

# Decision boundaries for linear SVM
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
plot_decision_boundary(
    linear_svm_moons, X_moon_train, y_moon_train,
    axes[0], title=f"LinearSVM — moons  test={acc_lin_moons_te:.2%}")
plot_decision_boundary(
    linear_svm_circles, X_circ_train, y_circ_train,
    axes[1], title=f"LinearSVM — circles  test={acc_lin_circ_te:.2%}")
plt.suptitle("Linear SVM Decision Boundaries", fontsize=12)
plt.tight_layout()
plt.show()

In [ ]:
# ── Train Kernel SVM (RBF) on both datasets ──────────────────────────────────
t0 = time.time()
kernel_svm_moons = KernelSVMScratch(
    C=C_KERNEL, kernel="rbf", gamma=GAMMA_RBF,
    learning_rate=LR_DUAL, n_iterations=N_ITER_DUAL)
kernel_svm_moons.fit(X_moon_train, y_moon_train)
t_kernel_moons = time.time() - t0

acc_rbf_moons_tr = kernel_svm_moons.score(X_moon_train, y_moon_train)
acc_rbf_moons_te = kernel_svm_moons.score(X_moon_test,  y_moon_test)

t0 = time.time()
kernel_svm_circles = KernelSVMScratch(
    C=C_KERNEL, kernel="rbf", gamma=GAMMA_RBF,
    learning_rate=LR_DUAL, n_iterations=N_ITER_DUAL)
kernel_svm_circles.fit(X_circ_train, y_circ_train)
t_kernel_circles = time.time() - t0

acc_rbf_circ_tr = kernel_svm_circles.score(X_circ_train, y_circ_train)
acc_rbf_circ_te = kernel_svm_circles.score(X_circ_test,  y_circ_test)

print("KernelSVMScratch (RBF) — make_moons")
print(f"  Train: {acc_rbf_moons_tr:.2%} | Test: {acc_rbf_moons_te:.2%} | "
      f"SVs: {kernel_svm_moons.n_support_}/{len(y_moon_train)} | Time: {t_kernel_moons:.2f}s")
print()
print("KernelSVMScratch (RBF) — make_circles")
print(f"  Train: {acc_rbf_circ_tr:.2%} | Test: {acc_rbf_circ_te:.2%} | "
      f"SVs: {kernel_svm_circles.n_support_}/{len(y_circ_train)} | Time: {t_kernel_circles:.2f}s")

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
plot_decision_boundary(
    kernel_svm_moons, X_moon_train, y_moon_train,
    axes[0], title=f"KernelSVM (RBF) — moons  test={acc_rbf_moons_te:.2%}", show_sv=True)
plot_decision_boundary(
    kernel_svm_circles, X_circ_train, y_circ_train,
    axes[1], title=f"KernelSVM (RBF) — circles  test={acc_rbf_circ_te:.2%}", show_sv=True)
plt.suptitle("Kernel SVM Decision Boundaries (support vectors circled in red)", fontsize=12)
plt.tight_layout()
plt.show()

### Library Comparison

We compare against sklearn's `LinearSVC` (liblinear primal solver) and `SVC`
(LibSVM — exact SMO dual solver). Our projected gradient ascent gives close but
not identical results; the SMO solver converges to the exact KKT point.

In [ ]:
# ── sklearn LinearSVC and SVC comparison ─────────────────────────────────────
sk_linear_moons = LinearSVC(C=C_LINEAR, max_iter=5000, random_state=SEED)
sk_linear_moons.fit(X_moon_train, y_moon_train)
sk_acc_lin_moons = sk_linear_moons.score(X_moon_test, y_moon_test)

sk_linear_circles = LinearSVC(C=C_LINEAR, max_iter=5000, random_state=SEED)
sk_linear_circles.fit(X_circ_train, y_circ_train)
sk_acc_lin_circles = sk_linear_circles.score(X_circ_test, y_circ_test)

sk_rbf_moons = SVC(C=C_KERNEL, kernel="rbf", gamma=GAMMA_RBF, random_state=SEED)
sk_rbf_moons.fit(X_moon_train, y_moon_train)
sk_acc_rbf_moons = sk_rbf_moons.score(X_moon_test, y_moon_test)

sk_rbf_circles = SVC(C=C_KERNEL, kernel="rbf", gamma=GAMMA_RBF, random_state=SEED)
sk_rbf_circles.fit(X_circ_train, y_circ_train)
sk_acc_rbf_circles = sk_rbf_circles.score(X_circ_test, y_circ_test)

print("Model Comparison — Test Accuracy")
print(f"{'Model':<42} {'moons':>8} {'circles':>9}")
print("-" * 62)
rows = [
    ("Majority-class baseline",            f"{baseline_moons:.2%}",   f"{baseline_circles:.2%}"),
    ("LinearSVM scratch (subgradient)",    f"{acc_lin_moons_te:.2%}", f"{acc_lin_circ_te:.2%}"),
    ("KernelSVM RBF scratch (proj. grad)", f"{acc_rbf_moons_te:.2%}", f"{acc_rbf_circ_te:.2%}"),
    ("sklearn LinearSVC (liblinear)",      f"{sk_acc_lin_moons:.2%}", f"{sk_acc_lin_circles:.2%}"),
    ("sklearn SVC RBF (LibSVM SMO)",       f"{sk_acc_rbf_moons:.2%}", f"{sk_acc_rbf_circles:.2%}"),
]
for name, m, c in rows:
    print(f"  {name:<40} {m:>8} {c:>9}")

print()
print(f"sklearn SV counts: moons={sk_rbf_moons.n_support_.sum()}, "
      f"circles={sk_rbf_circles.n_support_.sum()}")
print(f"scratch SV counts: moons={kernel_svm_moons.n_support_}, "
      f"circles={kernel_svm_circles.n_support_}")
print()
print("Note: Projected gradient ascent gives close results to exact SMO; differences")
print("      arise from approximation in the dual projection steps.")

---
## Part 4 — Evaluation & Analysis

### Decision Boundary Gallery — All Kernels

Three kernels on two datasets reveal each kernel\'s implicit feature space:
- **Linear:** Half-plane — fails on radially symmetric data.
- **Polynomial (degree 3):** Degree-$d$ algebraic boundary — intermediate.
- **RBF:** Smooth, radially-local boundary — best for both datasets.

In [ ]:
# ── Decision Boundary Gallery: 3 Kernels × 2 Datasets ────────────────────────
kernel_names = ["linear", "poly", "rbf"]
datasets_gallery = [
    (X_moon_train, y_moon_train, X_moon_test, y_moon_test, "make_moons"),
    (X_circ_train, y_circ_train, X_circ_test, y_circ_test, "make_circles"),
]

fig, axes = plt.subplots(2, 3, figsize=(14, 9))

for row, (X_tr, y_tr, X_te, y_te, ds_name) in enumerate(datasets_gallery):
    for col, kname in enumerate(kernel_names):
        ax = axes[row, col]
        clf = KernelSVMScratch(
            C=C_KERNEL, kernel=kname,
            gamma=GAMMA_RBF, degree=DEGREE_POLY, coef0=COEF0_POLY,
            learning_rate=LR_DUAL, n_iterations=N_ITER_DUAL,
        )
        clf.fit(X_tr, y_tr)
        te_acc = clf.score(X_te, y_te)
        title = (f"{kname.capitalize()} — {ds_name}\n"
                 f"test={te_acc:.2%}  SVs={clf.n_support_}")
        plot_decision_boundary(clf, X_tr, y_tr, ax, title=title, show_sv=True)

plt.suptitle("Kernel SVM — Decision Boundary Gallery (all kernels × both datasets)",
             fontsize=13, y=1.01)
plt.tight_layout()
plt.show()

print("Summary:")
print("  Linear: fails on circles (~50% = baseline), decent on moons.")
print("  Polynomial (d=3): captures mild curvature, better than linear.")
print("  RBF: best on both — local Gaussian influence adapts to any shape.")

### Ablation: Soft-Margin Parameter $C$

$C$ trades off margin width vs. hinge-loss violations:
- **Small $C$** → wide margin, more violations allowed (bias up, variance down).
- **Large $C$** → tight margin, fewer violations (overfitting risk, fewer SVs).

In [ ]:
# ── Ablation: C Parameter — KernelSVM (RBF) on make_moons ───────────────────
C_values: list[float] = [0.01, 0.1, 1.0, 10.0, 100.0]
c_train_accs: list[float] = []
c_test_accs:  list[float] = []
c_sv_counts:  list[int]   = []

for C_val in C_values:
    clf_c = KernelSVMScratch(
        C=C_val, kernel="rbf", gamma=GAMMA_RBF,
        learning_rate=LR_DUAL, n_iterations=N_ITER_DUAL,
    )
    clf_c.fit(X_moon_train, y_moon_train)
    c_train_accs.append(clf_c.score(X_moon_train, y_moon_train))
    c_test_accs.append(clf_c.score(X_moon_test,  y_moon_test))
    c_sv_counts.append(clf_c.n_support_)

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

axes[0].semilogx(C_values, c_train_accs, "o-", color="steelblue",  linewidth=2, label="Train")
axes[0].semilogx(C_values, c_test_accs,  "s--", color="darkorange", linewidth=2, label="Test")
axes[0].set_xlabel("C (log scale)")
axes[0].set_ylabel("Accuracy")
axes[0].set_title("Accuracy vs C — moons (RBF, gamma=1)")
axes[0].legend()
axes[0].set_ylim(0.45, 1.05)

axes[1].semilogx(C_values, c_sv_counts, "D-", color="firebrick", linewidth=2)
axes[1].set_xlabel("C (log scale)")
axes[1].set_ylabel("# Support Vectors")
axes[1].set_title("Support Vector Count vs C")

plt.suptitle("Soft-Margin C Ablation (RBF kernel, gamma=1.0, make_moons)", fontsize=12)
plt.tight_layout()
plt.show()

print(f"{'C':>8} | {'Train Acc':>10} | {'Test Acc':>9} | {'# SVs':>6}")
print("-" * 42)
for C_val, tr, te, sv in zip(C_values, c_train_accs, c_test_accs, c_sv_counts):
    print(f"{C_val:8.2f} | {tr:10.2%} | {te:9.2%} | {sv:6d}")
print("\nLarge C => fewer SVs (tight fit to each sample), risk of overfitting to noise.")

### Ablation: RBF Bandwidth $\gamma$

RBF kernel: $k(\mathbf{x}, \mathbf{x}') = \exp(-\gamma \|\mathbf{x} - \mathbf{x}'\|^2)$

- **Small $\gamma$** → wide Gaussians, smooth boundary, fewer SVs (underfitting risk).
- **Large $\gamma$** → narrow Gaussians, complex boundary, more SVs (overfitting risk).

In [ ]:
# ── Ablation: Gamma — KernelSVM (RBF) on make_circles ────────────────────────
gamma_values: list[float] = [0.1, 0.5, 1.0, 3.0, 10.0]
g_train_accs: list[float] = []
g_test_accs:  list[float] = []
g_sv_counts:  list[int]   = []

for g_val in gamma_values:
    clf_g = KernelSVMScratch(
        C=C_KERNEL, kernel="rbf", gamma=g_val,
        learning_rate=LR_DUAL, n_iterations=N_ITER_DUAL,
    )
    clf_g.fit(X_circ_train, y_circ_train)
    g_train_accs.append(clf_g.score(X_circ_train, y_circ_train))
    g_test_accs.append(clf_g.score(X_circ_test,   y_circ_test))
    g_sv_counts.append(clf_g.n_support_)

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

axes[0].semilogx(gamma_values, g_train_accs, "o-",  color="steelblue",  linewidth=2, label="Train")
axes[0].semilogx(gamma_values, g_test_accs,  "s--", color="darkorange", linewidth=2, label="Test")
axes[0].set_xlabel("gamma (log scale)")
axes[0].set_ylabel("Accuracy")
axes[0].set_title("Accuracy vs gamma — circles (RBF, C=1)")
axes[0].legend()
axes[0].set_ylim(0.45, 1.05)

axes[1].semilogx(gamma_values, g_sv_counts, "D-", color="firebrick", linewidth=2)
axes[1].set_xlabel("gamma (log scale)")
axes[1].set_ylabel("# Support Vectors")
axes[1].set_title("Support Vector Count vs gamma")

plt.suptitle("RBF Gamma Ablation (C=1.0, make_circles)", fontsize=12)
plt.tight_layout()
plt.show()

print(f"{'gamma':>8} | {'Train Acc':>10} | {'Test Acc':>9} | {'# SVs':>6}")
print("-" * 42)
for g, tr, te, sv in zip(gamma_values, g_train_accs, g_test_accs, g_sv_counts):
    print(f"{g:8.2f} | {tr:10.2%} | {te:9.2%} | {sv:6d}")
print("\nLarge gamma => train acc approaches 100%, but test acc drops (overfitting).")

### Support Vector Sparsity Analysis

The decision function is:
$f(\mathbf{x}) = \sum_{i \in \text{SVs}} \alpha_i y_i k(\mathbf{x}_i, \mathbf{x}) + b$

At prediction time we only evaluate the kernel against support vectors, not all
training points. This makes SVMs memory-efficient: if 10% of training samples
are SVs, inference is 10× faster than naive k-NN.

In [ ]:
# ── Support Vector Sparsity and Alpha Distribution ───────────────────────────

def compute_sv_stats(
    model: KernelSVMScratch,
    n_train: int,
) -> dict[str, float]:
    """Compute support vector statistics for a fitted KernelSVMScratch.

    Args:
        model:   Fitted KernelSVMScratch with alpha_ and sv_mask_ attributes.
        n_train: Number of training samples.

    Returns:
        Dictionary with n_sv, sv_fraction, mean_alpha, max_alpha.
    """
    sv_alphas = model.alpha_[model.sv_mask_]
    return {
        "n_sv":        float(model.n_support_),
        "sv_fraction": model.n_support_ / n_train,
        "mean_alpha":  float(sv_alphas.mean()) if len(sv_alphas) > 0 else 0.0,
        "max_alpha":   float(sv_alphas.max())  if len(sv_alphas) > 0 else 0.0,
    }


sv_stats_moons   = compute_sv_stats(kernel_svm_moons,   len(y_moon_train))
sv_stats_circles = compute_sv_stats(kernel_svm_circles, len(y_circ_train))

print(f"{'Metric':<20} {'make_moons':>12} {'make_circles':>14}")
print("-" * 48)
for key in sv_stats_moons:
    vm = sv_stats_moons[key]
    vc = sv_stats_circles[key]
    if key == "sv_fraction":
        print(f"{key:<20} {vm:>12.2%} {vc:>14.2%}")
    elif key == "n_sv":
        print(f"{key:<20} {int(vm):>12d} {int(vc):>14d}")
    else:
        print(f"{key:<20} {vm:>12.4f} {vc:>14.4f}")

# Alpha distribution histogram
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

for ax, model, ds_name in [
    (axes[0], kernel_svm_moons,   "make_moons"),
    (axes[1], kernel_svm_circles, "make_circles"),
]:
    sv_alphas = model.alpha_[model.sv_mask_]
    ax.hist(sv_alphas, bins=20, color="steelblue", edgecolor="white", alpha=0.85)
    ax.axvline(model.C, color="red", linestyle="--", linewidth=1.5,
               label=f"C = {model.C}")
    ax.set_xlabel("Alpha (dual variable)")
    ax.set_ylabel("Count")
    ax.set_title(f"Alpha Distribution — {ds_name}\n"
                 f"n_sv={model.n_support_}  ({sv_stats_moons['sv_fraction']:.1%} of train)")
    ax.legend(fontsize=9)

plt.suptitle("Support Vector Dual Variable (Alpha) Distributions", fontsize=12)
plt.tight_layout()
plt.show()

print("\nAlpha interpretation:")
print("  alpha = 0:        Non-support vector (margin >= 1, correctly classified)")
print("  0 < alpha < C:    Margin support vector (sits exactly on margin hyperplane)")
print("  alpha = C:        Bounded support vector (margin violated — in slack zone)")

In [ ]:
# ── Margin Distribution Analysis ─────────────────────────────────────────────
# The decision function value f(x) = sum_i alpha_i y_i K(x_i, x) + b
# measures signed distance to the decision boundary.
# Samples with |f(x)| > 1 are outside the margin (correctly classified freely).
# Samples with |f(x)| < 1 are inside the margin (support-vector territory).
# Samples with f(x) * y < 0 are misclassified.

def compute_margin_stats(
    model: object,
    X: np.ndarray,
    y: np.ndarray,
) -> dict[str, float]:
    """Compute margin distribution statistics for a fitted kernel SVM.

    Categorises training samples into three zones:
    - Free (correctly classified outside margin): y_i * f(x_i) > 1
    - Margin (support vector region): 0 < y_i * f(x_i) <= 1
    - Violated (misclassified): y_i * f(x_i) <= 0

    Args:
        model: Fitted classifier with a decision_function(X) method.
        X:     Feature matrix, shape (n_samples, n_features).
        y:     True labels in {0, 1}, shape (n_samples,).

    Returns:
        Dictionary with counts and fractions for each zone.
    """
    y_signed = 2 * y - 1  # {0,1} -> {-1,+1}
    dec_vals = model.decision_function(X)
    margins  = y_signed * dec_vals  # functional margins

    n_total    = len(y)
    n_free     = int(np.sum(margins >  1.0))
    n_margin   = int(np.sum((margins > 0.0) & (margins <= 1.0)))
    n_violated = int(np.sum(margins <= 0.0))

    return {
        "n_free":          n_free,
        "n_margin":        n_margin,
        "n_violated":      n_violated,
        "frac_free":       n_free     / n_total,
        "frac_margin":     n_margin   / n_total,
        "frac_violated":   n_violated / n_total,
        "mean_margin":     float(margins.mean()),
        "median_margin":   float(np.median(margins)),
    }


# Compute margin stats for all trained kernel SVMs
margin_stats_moons   = compute_margin_stats(kernel_svm_moons,   X_moon_train, y_moon_train)
margin_stats_circles = compute_margin_stats(kernel_svm_circles, X_circ_train, y_circ_train)

print("Margin Zone Analysis — Training Data")
print(f"{'Zone':<18} {'make_moons':>12} {'make_circles':>14}")
print("-" * 46)
for key in ["n_free", "n_margin", "n_violated", "frac_free", "frac_margin", "frac_violated"]:
    vm = margin_stats_moons[key]
    vc = margin_stats_circles[key]
    if "frac" in key:
        print(f"{key:<18} {vm:>12.2%} {vc:>14.2%}")
    else:
        print(f"{key:<18} {int(vm):>12d} {int(vc):>14d}")

# Margin distribution histogram
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

for ax, model, X_tr, y_tr, ds_name in [
    (axes[0], kernel_svm_moons,   X_moon_train, y_moon_train, "make_moons"),
    (axes[1], kernel_svm_circles, X_circ_train, y_circ_train, "make_circles"),
]:
    y_signed = 2 * y_tr - 1
    dec_vals = model.decision_function(X_tr)
    func_margins = y_signed * dec_vals

    ax.hist(func_margins, bins=30, color="steelblue", edgecolor="white", alpha=0.8,
            label="Functional margins")
    ax.axvline(0.0, color="red",    linestyle="--", linewidth=1.5, label="Decision boundary")
    ax.axvline(1.0, color="orange", linestyle=":",  linewidth=1.5, label="Margin +1")
    ax.axvline(-1e-6, color="gray", linestyle=":",  linewidth=1.0, alpha=0.5)

    stats = compute_margin_stats(model, X_tr, y_tr)
    ax.set_title(
        f"{ds_name}\n"
        f"Free={stats['frac_free']:.1%}  "
        f"Margin zone={stats['frac_margin']:.1%}  "
        f"Violated={stats['frac_violated']:.1%}",
        fontsize=10,
    )
    ax.set_xlabel("Functional margin y_i * f(x_i)")
    ax.set_ylabel("Count")
    ax.legend(fontsize=9)

plt.suptitle("Functional Margin Distribution — KernelSVM (RBF) on Training Data",
             fontsize=12)
plt.tight_layout()
plt.show()

print()
print("Interpretation:")
print("  Functional margin y_i * f(x_i) > 1:  Free — well beyond the margin.")
print("  0 < y_i * f(x_i) <= 1:               Margin zone — support-vector region.")
print("  y_i * f(x_i) <= 0:                   Violated — inside or wrong side.")
print("  The SVM minimises hinge loss, so violated + margin-zone samples drive training.")

In [ ]:
# ── Systematic Kernel Comparison Table ───────────────────────────────────────
# Train and evaluate all three kernels on both datasets, building a
# comprehensive comparison DataFrame with accuracy, F1, and SV count.

def evaluate_kernel_svm(
    kernel_name: str,
    X_train: np.ndarray,
    y_train: np.ndarray,
    X_test: np.ndarray,
    y_test: np.ndarray,
    C: float,
    gamma: float,
    degree: int,
    coef0: float,
    lr: float,
    n_iter: int,
) -> dict[str, object]:
    """Train a KernelSVMScratch and return evaluation metrics.

    Args:
        kernel_name: One of 'linear', 'rbf', 'poly'.
        X_train:     Training features, shape (n_train, d).
        y_train:     Training labels in {0, 1}, shape (n_train,).
        X_test:      Test features, shape (n_test, d).
        y_test:      Test labels in {0, 1}, shape (n_test,).
        C:           Soft-margin regularisation constant.
        gamma:       RBF bandwidth.
        degree:      Polynomial degree.
        coef0:       Polynomial constant.
        lr:          Dual gradient ascent learning rate.
        n_iter:      Number of optimisation iterations.

    Returns:
        Dictionary with kernel, train_acc, test_acc, test_f1, n_sv, sv_fraction.
    """
    clf = KernelSVMScratch(
        C=C, kernel=kernel_name, gamma=gamma, degree=degree,
        coef0=coef0, learning_rate=lr, n_iterations=n_iter,
    )
    t0 = time.time()
    clf.fit(X_train, y_train)
    train_time = time.time() - t0

    y_pred_tr = clf.predict(X_train)
    y_pred_te = clf.predict(X_test)

    return {
        "Kernel":       kernel_name,
        "Train Acc":    f"{float(np.mean(y_pred_tr == y_train)):.2%}",
        "Test Acc":     f"{float(np.mean(y_pred_te == y_test)):.2%}",
        "Test F1":      f"{f1_score(y_test, y_pred_te, average='macro'):.4f}",
        "# SVs":        clf.n_support_,
        "SV Fraction":  f"{clf.n_support_ / len(y_train):.2%}",
        "Time (s)":     f"{train_time:.2f}",
    }


# Run comparisons on both datasets
print("Kernel Comparison — make_moons")
print("-" * 70)
rows_moons = []
for kname in ["linear", "poly", "rbf"]:
    row = evaluate_kernel_svm(
        kname, X_moon_train, y_moon_train, X_moon_test, y_moon_test,
        C=C_KERNEL, gamma=GAMMA_RBF, degree=DEGREE_POLY, coef0=COEF0_POLY,
        lr=LR_DUAL, n_iter=N_ITER_DUAL,
    )
    rows_moons.append(row)

df_moons = pd.DataFrame(rows_moons)
print(df_moons.to_string(index=False))

print()
print("Kernel Comparison — make_circles")
print("-" * 70)
rows_circles = []
for kname in ["linear", "poly", "rbf"]:
    row = evaluate_kernel_svm(
        kname, X_circ_train, y_circ_train, X_circ_test, y_circ_test,
        C=C_KERNEL, gamma=GAMMA_RBF, degree=DEGREE_POLY, coef0=COEF0_POLY,
        lr=LR_DUAL, n_iter=N_ITER_DUAL,
    )
    rows_circles.append(row)

df_circles = pd.DataFrame(rows_circles)
print(df_circles.to_string(index=False))

print()
print("Takeaway: RBF dominates on both non-linear datasets.")
print("  Linear SVM on circles: ~50% (majority-class performance).")
print("  Polynomial (d=3): intermediate — captures some curvature but not radial symmetry.")
print("  SV fraction tracks boundary complexity: more complex boundary => more SVs.")

In [ ]:
# ── Error Analysis: Confusion Matrices and Misclassified Samples ─────────────
fig, axes = plt.subplots(1, 2, figsize=(10, 4))

for ax, model, X_te, y_te, ds_name in [
    (axes[0], kernel_svm_moons,   X_moon_test, y_moon_test,   "make_moons"),
    (axes[1], kernel_svm_circles, X_circ_test, y_circ_test,   "make_circles"),
]:
    y_pred = model.predict(X_te)
    cm = confusion_matrix(y_te, y_pred)
    ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=[0, 1]).plot(
        ax=ax, colorbar=False)
    ax.set_title(f"{ds_name}  Acc={accuracy_score(y_te, y_pred):.2%}", fontsize=11)

plt.suptitle("Confusion Matrices — KernelSVM (RBF, scratch)", fontsize=12)
plt.tight_layout()
plt.show()

# Visualise errors overlaid on the decision boundary
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

for ax, model, X_tr, y_tr, X_te, y_te, ds_name in [
    (axes[0], kernel_svm_moons,   X_moon_train, y_moon_train,
     X_moon_test, y_moon_test,   "make_moons"),
    (axes[1], kernel_svm_circles, X_circ_train, y_circ_train,
     X_circ_test, y_circ_test,   "make_circles"),
]:
    plot_decision_boundary(model, X_tr, y_tr, ax, title=ds_name, h=0.04)
    y_pred = model.predict(X_te)
    errors = X_te[y_pred != y_te]
    palette = ["steelblue" if c == 0 else "darkorange" for c in y_te]
    ax.scatter(X_te[:, 0], X_te[:, 1],
               c=palette, s=30, edgecolors="k", linewidths=0.4,
               alpha=0.9, zorder=4, label="Test")
    if len(errors) > 0:
        ax.scatter(errors[:, 0], errors[:, 1],
                   c="red", marker="x", s=90, linewidths=2.0,
                   zorder=5, label=f"Errors ({len(errors)})")
    ax.legend(fontsize=9)

plt.suptitle("Test Errors (red X) on Decision Boundary — KernelSVM (RBF)", fontsize=12)
plt.tight_layout()
plt.show()

print("Error analysis:")
for model, X_te, y_te, ds_name in [
    (kernel_svm_moons,   X_moon_test, y_moon_test,   "make_moons"),
    (kernel_svm_circles, X_circ_test, y_circ_test,   "make_circles"),
]:
    y_pred = model.predict(X_te)
    err_mask = y_pred != y_te
    n_errors = int(err_mask.sum())
    if n_errors > 0:
        dec_err = np.abs(model.decision_function(X_te[err_mask]))
        print(f"  {ds_name}: {n_errors} errors | "
              f"mean |decision score| at errors = {dec_err.mean():.4f} "
              f"(low score => decision boundary uncertainty)")

---
## Part 5 — Summary & Lessons Learned

### Key Takeaways

1. **Maximum-margin principle:** SVMs find the hyperplane that maximises the
   geometric margin $2/\|\mathbf{w}\|$. This is equivalent to L2 regularisation
   of the weight vector, giving SVMs a natural resistance to overfitting.

2. **Soft-margin $C$ trades bias for variance:** Small $C$ → wide margin →
   more hinge violations → higher bias but lower variance. Support vector count
   *decreases* as $C$ grows (fewer points are "hard" enough to influence the boundary).

3. **The kernel trick:** Replacing inner products with $k(\mathbf{x}_i, \mathbf{x}_j)$
   implicitly maps data into a (potentially infinite-dimensional) feature space
   without computing $\phi(\mathbf{x})$ explicitly. The RBF kernel makes concentric
   circles perfectly separable; no linear SVM can.

4. **Support vector sparsity:** Only samples with $\alpha_i > 0$ determine the
   boundary. Prediction scales with the number of SVs, not dataset size — unlike
   k-NN (which stores all points) or parametric models (which store weight vectors).

5. **Dual formulation enables kernels:** The Lagrangian dual depends only on kernel
   evaluations $k(\mathbf{x}_i, \mathbf{x}_j)$. Exact SMO solves this dual by
   updating two $\alpha_i$ at a time; our projected gradient ascent is approximate
   but pedagogically transparent.

### What's Next

→ **02-06 (Generalised Linear Models)** provides the theoretical framework
unifying linear regression (02-01), logistic regression (02-02), and softmax
as members of the exponential family — the "why" behind the loss functions we
have been minimising throughout this module.

### Going Further

- **SMO algorithm** (Platt 1998): Solves the dual exactly by analytically
  optimising two $\alpha$ variables at a time — the engine behind LibSVM.
- **Kernel PCA:** Eigendecomposition of the Gram matrix gives non-linear PCA;
  see Module 3-08 for a full treatment.
- **Platt scaling:** Fits a sigmoid on SVM decision scores to produce calibrated
  probabilities when `predict_proba` is needed downstream.
- **RBF Networks:** The RBF kernel SVM is related to a radial basis function
  network where each support vector is a basis function centre.